# 02 - HydroServer Bulk Loading Demo

## Setup and Creation Controls
This notebook creates one thing and two datastreams for every eligible station, then uploads Hydroweb and GEOGLOWS observations. Use API-key authentication only in a disposable demo workspace.

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd

try:
    from hydroserverpy import HydroServer
except Exception as exc:
    HydroServer = None
    print(f"hydroserverpy is not available yet: {exc}")

HYDROSERVER_HOST = "https://playground.hydroserver.org"
WORKSPACE_NAME = "hydroserver_uganda_demo"
WORKSPACE_IS_PRIVATE = False

# Choose one: "anonymous" or "api_key".
AUTH_METHOD = "api_key"
HYDROSERVER_API_KEY = ""  # Keep secrets out of saved notebooks. Paste only when prompted.

# Facilitator controls. Defaults keep notebooks safe for anonymous/local runs.
CREATE_WORKSPACE_IF_MISSING = False
DELETE_CREATED_RESOURCES_AT_END = False
DEMO_RESOURCE_PREFIX = "Uganda Demo"
DEMO_RUN_SUFFIX = ""

# Google Colab/local path support. Leave blank unless data is somewhere custom.
DATA_DIR_OVERRIDE = ""


def resolve_data_dir(data_dir_override=""):
    candidates = []
    if data_dir_override:
        candidates.append(Path(data_dir_override).expanduser())
    candidates.extend([
        Path("data"),
        Path("../data"),
        Path("hydroserver_workshop/data"),
        Path("../hydroserver_workshop/data"),
        Path("/content/hydroserver_workshop/data"),
        Path("/content/data"),
    ])
    for candidate in candidates:
        if candidate.exists() and (candidate / "Uganda_Hydroweb.csv").exists():
            return candidate
    searched = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the workshop data folder. Upload hydroserver_workshop/data, "
        "upload data/ beside the notebook, or set DATA_DIR_OVERRIDE. Searched:\n"
        f"{searched}"
    )


DATA_DIR = resolve_data_dir(DATA_DIR_OVERRIDE)
STATION_CATALOG_CSV = DATA_DIR / "Uganda_Hydroweb.csv"
SELECTED_STATION_CSV = DATA_DIR / "uganda_selected_station.csv"
HYDROWEB_DIR = DATA_DIR / "hydroweb"
GEOGLOWS_DIR = DATA_DIR / "geoglows"
STREAMFLOW_CSV = DATA_DIR / "sample_streamflow_observations.csv"
FORECAST_CSV = DATA_DIR / "sample_forecast_timeseries.csv"

demo_run_suffix = DEMO_RUN_SUFFIX or datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
demo_resource_prefix = f"{DEMO_RESOURCE_PREFIX} {demo_run_suffix}"

print(f"HydroServer host: {HYDROSERVER_HOST}")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Authentication mode: {AUTH_METHOD}")
print(f"Data directory: {DATA_DIR}")

MAX_STATIONS_TO_LOAD = None  # Keep None to load every eligible station. Set a small integer for a quick rehearsal.

HydroServer host: https://playground.hydroserver.org
Workspace: hydroserver_uganda_demo
Authentication mode: api_key
Data directory: ../data


## Connect to HydroServer

In [4]:
def _prompt_if_needed(value, prompt):
    return value if value else getpass(prompt)

hs_api = None

if HydroServer is None:
    print("Install hydroserverpy before connecting to HydroServer.")
elif AUTH_METHOD == "anonymous":
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST)
        print("Connected anonymously. Anonymous mode can read public data but cannot create or upload resources.")
    except Exception as exc:
        print(f"Anonymous connection failed: {exc}")
elif AUTH_METHOD == "api_key":
    api_key = _prompt_if_needed(HYDROSERVER_API_KEY, "HydroServer API key: ")
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST, apikey=api_key)
        print("Connected with API-key authentication.")
    except Exception as exc:
        print(f"API-key connection failed: {exc}")
else:
    raise ValueError("AUTH_METHOD must be 'anonymous' or 'api_key'.")

Connected with API-key authentication.


## Track Created Resources

In [5]:
created_resources = []


def resource_uid(resource):
    if resource is None:
        return None
    if isinstance(resource, str):
        return resource
    if isinstance(resource, dict):
        for key in ("uid", "id", "workspace_id"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("uid", "id", "workspace_id"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_uid(dumped)
    return None


def resource_name(resource):
    if resource is None:
        return None
    if isinstance(resource, dict):
        for key in ("name", "code", "definition", "sampling_feature_code"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("name", "code", "definition", "sampling_feature_code"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_name(dumped)
    return type(resource).__name__


def record_resource(resource_type, resource, station_id=None, source=None):
    created_resources.append({
        "resource_type": resource_type,
        "station_id": station_id,
        "source": source,
        "name_or_code": resource_name(resource),
        "uuid": resource_uid(resource),
        "python_type": type(resource).__name__,
        "resource": resource,
    })
    print(f"{resource_type}: {resource_name(resource)} | uuid={resource_uid(resource)}")
    return resource


def created_resources_dataframe(include_objects=False):
    rows = []
    for row in created_resources:
        rows.append({key: value for key, value in row.items() if include_objects or key != "resource"})
    return pd.DataFrame(rows)

print("Resource registry initialized.")

Resource registry initialized.


## Find or Optionally Create Demo Workspace

In [6]:
workspace = None
workspace_uid = None

if hs_api is None:
    print("Skipping workspace lookup because the HydroServer client is unavailable.")
elif AUTH_METHOD == "anonymous":
    print(f"Anonymous mode: cannot create or manage workspace '{WORKSPACE_NAME}'.")
    print("Switch AUTH_METHOD to 'api_key' for live creation/upload demos.")
else:
    try:
        workspaces = hs_api.workspaces.list(fetch_all=True)
        workspace_items = getattr(workspaces, "items", workspaces)
        workspace = next((item for item in workspace_items if getattr(item, "name", None) == WORKSPACE_NAME), None)
        if workspace is None and CREATE_WORKSPACE_IF_MISSING:
            workspace = record_resource(
                "workspace",
                hs_api.workspaces.create(name=WORKSPACE_NAME, is_private=WORKSPACE_IS_PRIVATE),
            )
        elif workspace is None:
            print(f"Workspace '{WORKSPACE_NAME}' was not found. Ask the facilitator to create it first.")
        else:
            print(f"Using existing workspace: {getattr(workspace, 'name', WORKSPACE_NAME)}")
        workspace_uid = resource_uid(workspace)
        if workspace_uid:
            print(f"Workspace UUID: {workspace_uid}")
    except Exception as exc:
        print(f"Could not find or create workspace '{WORKSPACE_NAME}': {exc}")

Using existing workspace: hydroserver_uganda_demo
Workspace UUID: 019dcc03-9020-718b-9c66-d9da3401eede


## Load Station Files and Build Bulk Plan

In [7]:
def identifier_text(value):
    if pd.isna(value):
        return ""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def station_file_index(directory):
    if not directory.exists():
        return {}
    return {path.stem: path for path in directory.glob("*.csv")}


def resolve_station_file(index, station, identifiers):
    for identifier in identifiers:
        key = identifier_text(identifier)
        if key and key in index:
            return index[key]
    return None


def normalize_hydroweb_water_level(path, station):
    frame = pd.read_csv(path)
    if list(frame.columns) != ["Datetime", "Water Level (m)"]:
        raise ValueError(f"{path.name} should have columns: Datetime, Water Level (m)")
    return pd.DataFrame({
        "phenomenon_time": pd.to_datetime(frame["Datetime"], utc=True),
        "result": pd.to_numeric(frame["Water Level (m)"], errors="raise"),
        "source": "Hydroweb/Theia",
        "source_identifier": identifier_text(station["COMID_v1"]),
        "station_id": station["ID"],
        "observed_property": "Water Level",
        "unit": "m",
    }).dropna(subset=["phenomenon_time", "result"]).reset_index(drop=True)


def normalize_two_column_geoglows(path, station):
    frame = pd.read_csv(path)
    if len(frame.columns) != 2:
        raise ValueError(f"{path.name} should have exactly two columns")
    timestamp_column, value_column = frame.columns
    return pd.DataFrame({
        "phenomenon_time": pd.to_datetime(frame[timestamp_column], utc=True),
        "result": pd.to_numeric(frame[value_column], errors="raise"),
        "source": "GEOGLOWS",
        "source_identifier": identifier_text(station["COMID_v2"]),
        "station_id": station["ID"],
        "observed_property": "Streamflow",
        "unit": "m3/s",
    }).dropna(subset=["phenomenon_time", "result"]).reset_index(drop=True)

station_catalog = pd.read_csv(STATION_CATALOG_CSV)
selected_station = pd.read_csv(SELECTED_STATION_CSV).iloc[0]
hydroweb_index = station_file_index(HYDROWEB_DIR)
geoglows_index = station_file_index(GEOGLOWS_DIR)

bulk_rows = []
for _, station in station_catalog.iterrows():
    hydroweb_file = resolve_station_file(hydroweb_index, station, [station["ID"], station["COMID_v1"]])
    geoglows_file = resolve_station_file(geoglows_index, station, [station["ID"], station["COMID_v2"]])
    has_comids = identifier_text(station["COMID_v1"]) not in ("", "0") and identifier_text(station["COMID_v2"]) not in ("", "0")
    skip_reasons = []
    if not has_comids:
        skip_reasons.append("missing COMID_v1 or COMID_v2")
    if hydroweb_file is None:
        skip_reasons.append("missing Hydroweb CSV")
    if geoglows_file is None:
        skip_reasons.append("missing GEOGLOWS CSV")
    bulk_rows.append({
        "station_id": station["ID"],
        "name": station["Name"],
        "river": station["River"],
        "COMID_v1": identifier_text(station["COMID_v1"]),
        "COMID_v2": identifier_text(station["COMID_v2"]),
        "hydroweb_file": str(hydroweb_file) if hydroweb_file else "",
        "geoglows_file": str(geoglows_file) if geoglows_file else "",
        "is_loadable": not skip_reasons,
        "skip_reason": "; ".join(skip_reasons),
    })

bulk_station_load_plan = pd.DataFrame(bulk_rows)
loadable_station_plan = bulk_station_load_plan[bulk_station_load_plan["is_loadable"]].copy()
if MAX_STATIONS_TO_LOAD is not None:
    loadable_station_plan = loadable_station_plan.head(MAX_STATIONS_TO_LOAD)

summary = pd.DataFrame([{
    "stations_in_catalog": len(station_catalog),
    "hydroweb_station_files": len(hydroweb_index),
    "geoglows_station_files": len(geoglows_index),
    "eligible_stations": int(bulk_station_load_plan["is_loadable"].sum()),
    "stations_selected_for_this_run": len(loadable_station_plan),
}])
display(summary)
display(loadable_station_plan.head(10))

selected_hydroweb_file = resolve_station_file(hydroweb_index, selected_station, [selected_station["ID"], selected_station["COMID_v1"]])
selected_geoglows_file = resolve_station_file(geoglows_index, selected_station, [selected_station["ID"], selected_station["COMID_v2"]])
hydroweb_water_level_payload = normalize_hydroweb_water_level(selected_hydroweb_file, selected_station)
geoglows_streamflow_payload = normalize_two_column_geoglows(selected_geoglows_file, selected_station)
print(f"Selected station: {selected_station['ID']} / {selected_station['Name']}")
print(f"Hydroweb rows: {len(hydroweb_water_level_payload)}")
print(f"GEOGLOWS rows: {len(geoglows_streamflow_payload)}")
display(hydroweb_water_level_payload.head())
display(geoglows_streamflow_payload.head())

,stations_in_catalog,hydroweb_station_files,geoglows_station_files,eligible_stations,stations_selected_for_this_run
0,101,75,49,38,38


,station_id,name,river,COMID_v1,COMID_v2,hydroweb_file,geoglows_file,is_loadable,skip_reason
4,H-102549,Nile_achwa_km5268,Achwa,7068385,160214697,../data/hydroweb/H-102549.csv,../data/geoglows/160214697.csv,True,
10,H-107870,Nile_akagera_km6130,Akagera,7073854,160220651,../data/hydroweb/H-107870.csv,../data/geoglows/160220651.csv,True,
11,H-0000000007850,Nile_akagera_km6148,Akagera,7073854,160214812,../data/hydroweb/H-0000000007850.csv,../data/geoglows/160214812.csv,True,
12,H-101620,Nile_akokoro_km5900,Akokoro,7069648,160214717,../data/hydroweb/H-101620.csv,../data/geoglows/160214717.csv,True,
21,H-100136,Nile_kafu_km5582,Kafu,7070059,160207720,../data/hydroweb/H-100136.csv,../data/geoglows/160207720.csv,True,
24,H-107033,Nile_kafu_km5672,Kafu,7070519,160213572,../data/hydroweb/H-107033.csv,../data/geoglows/160213572.csv,True,
39,H-107872,Nile_lugogo-trib01_km5762,Lugogo-trib01,7071031,160229935,../data/hydroweb/H-107872.csv,../data/geoglows/160229935.csv,True,
40,H-109552,Nile_malaba_km6001,Malaba,7071495,160279000,../data/hydroweb/H-109552.csv,../data/geoglows/160279000.csv,True,
41,H-106712,Nile_malaba-trib01_km5860,Malaba-trib01,7071533,160250965,../data/hydroweb/H-106712.csv,../data/geoglows/160250965.csv,True,
42,H-101618,Nile_manafwa_km5920,Manafwa,7070939,160231100,../data/hydroweb/H-101618.csv,../data/geoglows/160231100.csv,True,


Selected station: H-102549 / Nile_achwa_km5268
Hydroweb rows: 63
GEOGLOWS rows: 31523


,phenomenon_time,result,source,source_identifier,station_id,observed_property,unit
0,2020-09-02 00:00:00+00:00,996.26,Hydroweb/Theia,7068385,H-102549,Water Level,m
1,2020-10-26 00:00:00+00:00,996.93,Hydroweb/Theia,7068385,H-102549,Water Level,m
2,2020-11-22 00:00:00+00:00,994.76,Hydroweb/Theia,7068385,H-102549,Water Level,m
3,2020-12-19 00:00:00+00:00,993.66,Hydroweb/Theia,7068385,H-102549,Water Level,m
4,2021-01-15 00:00:00+00:00,993.04,Hydroweb/Theia,7068385,H-102549,Water Level,m


,phenomenon_time,result,source,source_identifier,station_id,observed_property,unit
0,1940-01-01 00:00:00+00:00,2.812,GEOGLOWS,160214697,H-102549,Streamflow,m3/s
1,1940-01-02 00:00:00+00:00,6.470,GEOGLOWS,160214697,H-102549,Streamflow,m3/s
2,1940-01-03 00:00:00+00:00,9.973,GEOGLOWS,160214697,H-102549,Streamflow,m3/s
3,1940-01-04 00:00:00+00:00,15.123,GEOGLOWS,160214697,H-102549,Streamflow,m3/s
4,1940-01-05 00:00:00+00:00,19.487,GEOGLOWS,160214697,H-102549,Streamflow,m3/s


## Create Needed Metadata, Things, and Datastreams

In [8]:
selected_station = pd.read_csv(SELECTED_STATION_CSV).iloc[0]


def optional_float(value, default=None):
    if pd.isna(value):
        return default
    return float(value)


def build_thing_template(station):
    return {
        "name": f"{demo_resource_prefix} {station['ID']} {station.get('Name', 'Uganda Station')}",
        "description": "Uganda Hydroweb/GEOGLOWS station used for workshop observations.",
        "sampling_feature_type": "Site",
        "sampling_feature_code": str(station["ID"]),
        "site_type": "Stream",
        "latitude": optional_float(station["Latitude"]),
        "longitude": optional_float(station["Longitude"]),
        "elevation_m": optional_float(station.get("Elevation"), 0.0),
        "elevation_datum": str(station.get("Ellipsoid", "WGS84")),
        "state": "",
        "county": str(station.get("River", "")),
        "country": "UG",
        "data_disclaimer": "Workshop demonstration data from local Hydroweb and GEOGLOWS CSV files.",
        "is_private": False,
        "workspace": workspace_uid or "<workspace-uuid>",
    }

water_level_observed_property_template = {
    "name": f"{demo_resource_prefix} Water Level",
    "definition": "Satellite altimetry water level",
    "description": "Hydroweb/Theia water level for Uganda stations.",
    "observed_property_type": "Hydrology",
    "code": f"WaterLevel_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
streamflow_observed_property_template = {
    "name": f"{demo_resource_prefix} Streamflow",
    "definition": "Water discharge in a river channel",
    "description": "GEOGLOWS streamflow for Uganda stations.",
    "observed_property_type": "Hydrology",
    "code": f"Streamflow_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
water_level_unit_template = {
    "name": f"{demo_resource_prefix} Meter",
    "symbol": "m",
    "definition": "Meter",
    "unit_type": "Length",
    "workspace": workspace_uid or "<workspace-uuid>",
}
streamflow_unit_template = {
    "name": f"{demo_resource_prefix} Cubic meters per second",
    "symbol": "m3/s",
    "definition": "Cubic meters per second",
    "unit_type": "Discharge",
    "workspace": workspace_uid or "<workspace-uuid>",
}
hydroweb_sensor_template = {
    "name": f"{demo_resource_prefix} Hydroweb Theia Altimetry",
    "description": "Hydroweb/Theia satellite altimetry water-level source.",
    "encoding_type": "application/json",
    "manufacturer": "Theia Hydroweb",
    "sensor_model": str(selected_station.get("Missions", "Hydroweb")),
    "sensor_model_link": "https://catalogue.theia.data-terra.org/collection/HYDROWEB_RIVERS_OPE",
    "method_type": "Satellite altimetry",
    "method_link": "https://catalogue.theia.data-terra.org/collection/HYDROWEB_RIVERS_OPE",
    "method_code": f"HYDROWEB_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
geoglows_sensor_template = {
    "name": f"{demo_resource_prefix} GEOGLOWS RFS",
    "description": "GEOGLOWS modeled streamflow source.",
    "encoding_type": "application/json",
    "manufacturer": "GEOGLOWS",
    "sensor_model": "GEOGLOWS RFS",
    "sensor_model_link": "https://data.geoglows.org/",
    "method_type": "Model",
    "method_link": "https://data.geoglows.org/",
    "method_code": f"GEOGLOWS_{demo_run_suffix}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
processing_level_template = {
    "code": f"RAW_{demo_run_suffix}",
    "definition": "Raw",
    "explanation": "Data have not been processed or quality controlled in the workshop.",
    "workspace": workspace_uid or "<workspace-uuid>",
}
result_qualifier_template = {
    "code": f"SUSPECT_{demo_run_suffix}",
    "description": "Observation should be reviewed before operational use.",
    "workspace": workspace_uid or "<workspace-uuid>",
}

metadata_templates = pd.DataFrame([
    {"resource_type": "thing", "name_or_code": build_thing_template(selected_station)["name"]},
    {"resource_type": "observed_property", "name_or_code": water_level_observed_property_template["code"]},
    {"resource_type": "observed_property", "name_or_code": streamflow_observed_property_template["code"]},
    {"resource_type": "unit", "name_or_code": water_level_unit_template["symbol"]},
    {"resource_type": "unit", "name_or_code": streamflow_unit_template["symbol"]},
    {"resource_type": "sensor", "name_or_code": hydroweb_sensor_template["method_code"]},
    {"resource_type": "sensor", "name_or_code": geoglows_sensor_template["method_code"]},
    {"resource_type": "processing_level", "name_or_code": processing_level_template["code"]},
    {"resource_type": "result_qualifier", "name_or_code": result_qualifier_template["code"]},
])
display(metadata_templates)


def build_datastream_template(name, description, thing, sensor, observed_property, processing_level, unit, begin_time):
    return {
        "name": name,
        "description": description,
        "observation_type": "Field Observation",
        "sampled_medium": "Water",
        "no_data_value": -9999,
        "aggregation_statistic": "Continuous",
        "time_aggregation_interval": 1,
        "status": "Ongoing",
        "result_type": "Timeseries",
        "value_count": 0,
        "phenomenon_begin_time": begin_time,
        "phenomenon_end_time": None,
        "result_begin_time": begin_time,
        "result_end_time": None,
        "is_visible": True,
        "is_private": False,
        "thing": resource_uid(thing),
        "sensor": resource_uid(sensor),
        "observed_property": resource_uid(observed_property),
        "processing_level": resource_uid(processing_level),
        "unit": resource_uid(unit),
        "time_aggregation_interval_unit": "hours",
        "intended_time_spacing": 1,
        "intended_time_spacing_unit": "days",
    }

shared = {}
station_datastreams = []

if hs_api is None or workspace is None or workspace_uid is None:
    print("Skipping creation because an authenticated workspace is required.")
elif loadable_station_plan.empty:
    print("No stations are eligible for bulk loading.")
else:
    try:
        shared["water_level_observed_property"] = record_resource("observed_property", hs_api.observedproperties.create(**water_level_observed_property_template))
        shared["streamflow_observed_property"] = record_resource("observed_property", hs_api.observedproperties.create(**streamflow_observed_property_template))
        shared["water_level_unit"] = record_resource("unit", hs_api.units.create(**water_level_unit_template))
        shared["streamflow_unit"] = record_resource("unit", hs_api.units.create(**streamflow_unit_template))
        shared["hydroweb_sensor"] = record_resource("sensor", hs_api.sensors.create(**hydroweb_sensor_template))
        shared["geoglows_sensor"] = record_resource("sensor", hs_api.sensors.create(**geoglows_sensor_template))
        shared["processing_level"] = record_resource("processing_level", hs_api.processinglevels.create(**processing_level_template))

        for _, plan_row in loadable_station_plan.iterrows():
            station = station_catalog[station_catalog["ID"] == plan_row["station_id"]].iloc[0]
            hydroweb_payload = normalize_hydroweb_water_level(Path(plan_row["hydroweb_file"]), station)
            geoglows_payload = normalize_two_column_geoglows(Path(plan_row["geoglows_file"]), station)
            thing = record_resource("thing", hs_api.things.create(**build_thing_template(station)), station_id=station["ID"])
            hydroweb_ds = record_resource(
                "datastream",
                hs_api.datastreams.create(**build_datastream_template(
                    name=f"{demo_resource_prefix} {station['ID']} Hydroweb Water Level",
                    description="Hydroweb/Theia water-level observations loaded from local CSV.",
                    thing=thing,
                    sensor=shared["hydroweb_sensor"],
                    observed_property=shared["water_level_observed_property"],
                    processing_level=shared["processing_level"],
                    unit=shared["water_level_unit"],
                    begin_time=hydroweb_payload["phenomenon_time"].min().to_pydatetime(),
                )),
                station_id=station["ID"],
                source="Hydroweb/Theia",
            )
            geoglows_ds = record_resource(
                "datastream",
                hs_api.datastreams.create(**build_datastream_template(
                    name=f"{demo_resource_prefix} {station['ID']} GEOGLOWS Streamflow",
                    description="GEOGLOWS streamflow observations loaded from local CSV.",
                    thing=thing,
                    sensor=shared["geoglows_sensor"],
                    observed_property=shared["streamflow_observed_property"],
                    processing_level=shared["processing_level"],
                    unit=shared["streamflow_unit"],
                    begin_time=geoglows_payload["phenomenon_time"].min().to_pydatetime(),
                )),
                station_id=station["ID"],
                source="GEOGLOWS",
            )
            station_datastreams.append({"station_id": station["ID"], "hydroweb": hydroweb_ds, "geoglows": geoglows_ds})
    except Exception as exc:
        print(f"Could not create bulk resources: {exc}")

display(created_resources_dataframe())

,resource_type,name_or_code
0,thing,Uganda Demo 20260427220902 H-102549 Nile_achwa...
1,observed_property,WaterLevel_20260427220902
2,observed_property,Streamflow_20260427220902
3,unit,m
4,unit,m3/s
5,sensor,HYDROWEB_20260427220902
6,sensor,GEOGLOWS_20260427220902
7,processing_level,RAW_20260427220902
8,result_qualifier,SUSPECT_20260427220902


observed_property: Uganda Demo 20260427220902 Water Level | uuid=019dd0fe-1961-79e5-bca8-5c8604064b3c
observed_property: Uganda Demo 20260427220902 Streamflow | uuid=019dd0fe-1ccb-756d-93a8-a366f96df79d
unit: Uganda Demo 20260427220902 Meter | uuid=019dd0fe-203a-72d4-832a-efa3f737361b
unit: Uganda Demo 20260427220902 Cubic meters per second | uuid=019dd0fe-23a0-7fb3-b4b0-321f7f75e6fc
sensor: Uganda Demo 20260427220902 Hydroweb Theia Altimetry | uuid=019dd0fe-270a-7104-b6d9-5a4dd83a6b5e
sensor: Uganda Demo 20260427220902 GEOGLOWS RFS | uuid=019dd0fe-2a76-78cf-821b-3c161bc9fc1d
processing_level: RAW_20260427220902 | uuid=019dd0fe-2dd3-7c7c-a985-79bac97c85ad
thing: Uganda Demo 20260427220902 H-102549 Nile_achwa_km5268 | uuid=019dd0fe-3148-7ed1-9f58-3c954df234c1
datastream: Uganda Demo 20260427220902 H-102549 Hydroweb Water Level | uuid=019dd0fe-33e4-7761-a045-e5aae79620ce
datastream: Uganda Demo 20260427220902 H-102549 GEOGLOWS Streamflow | uuid=019dd0fe-3680-78ff-8206-331ee304f580
thing:

,resource_type,station_id,source,name_or_code,uuid,python_type
0,observed_property,NaN,NaN,Uganda Demo 20260427220902 Water Level,019dd0fe-1961-79e5-bca8-5c8604064b3c,ObservedProperty
1,observed_property,NaN,NaN,Uganda Demo 20260427220902 Streamflow,019dd0fe-1ccb-756d-93a8-a366f96df79d,ObservedProperty
2,unit,NaN,NaN,Uganda Demo 20260427220902 Meter,019dd0fe-203a-72d4-832a-efa3f737361b,Unit
3,unit,NaN,NaN,Uganda Demo 20260427220902 Cubic meters per se...,019dd0fe-23a0-7fb3-b4b0-321f7f75e6fc,Unit
4,sensor,NaN,NaN,Uganda Demo 20260427220902 Hydroweb Theia Alti...,019dd0fe-270a-7104-b6d9-5a4dd83a6b5e,Sensor
...,...,...,...,...,...,...
116,datastream,H-0000000007959,Hydroweb/Theia,Uganda Demo 20260427220902 H-0000000007959 Hyd...,019dd0ff-a442-7102-a97f-1e128be39678,Datastream
117,datastream,H-0000000007959,GEOGLOWS,Uganda Demo 20260427220902 H-0000000007959 GEO...,019dd0ff-a7cd-7733-aba6-c9ad4673ac57,Datastream
118,thing,H-108720,NaN,Uganda Demo 20260427220902 H-108720 Nile_victo...,019dd0ff-aa7a-78e9-814f-dcdca7a959da,Thing
119,datastream,H-108720,Hydroweb/Theia,Uganda Demo 20260427220902 H-108720 Hydroweb W...,019dd0ff-adf3-7a76-872d-cc62d881676e,Datastream


## Upload Hydroweb and GEOGLOWS Observations

In [9]:
upload_report_rows = []

if hs_api is None:
    print("Skipping upload because the HydroServer client is unavailable.")
elif not station_datastreams:
    print("Skipping upload because no datastreams were created in this run.")
else:
    try:
        for ds_row in station_datastreams:
            station = station_catalog[station_catalog["ID"] == ds_row["station_id"]].iloc[0]
            plan_row = loadable_station_plan[loadable_station_plan["station_id"] == ds_row["station_id"]].iloc[0]
            payload_pairs = [
                ("Hydroweb/Theia", ds_row["hydroweb"], normalize_hydroweb_water_level(Path(plan_row["hydroweb_file"]), station)),
                ("GEOGLOWS", ds_row["geoglows"], normalize_two_column_geoglows(Path(plan_row["geoglows_file"]), station)),
            ]
            for source, datastream, payload in payload_pairs:
                upload_payload = payload[["phenomenon_time", "result"]].copy()
                datastream.load_observations(upload_payload)
                upload_report_rows.append({
                    "station_id": ds_row["station_id"],
                    "source": source,
                    "datastream_uuid": resource_uid(datastream),
                    "rows_uploaded": len(upload_payload),
                    "first_time": upload_payload["phenomenon_time"].min(),
                    "last_time": upload_payload["phenomenon_time"].max(),
                })
                print(f"Uploaded {len(upload_payload)} rows for {ds_row['station_id']} {source}.")
    except Exception as exc:
        print(f"Could not upload observations: {exc}")

upload_report = pd.DataFrame(upload_report_rows)
if not upload_report.empty:
    display(upload_report)
else:
    print("No observations were uploaded in this run.")

Uploaded 63 rows for H-102549 Hydroweb/Theia.
Uploaded 31523 rows for H-102549 GEOGLOWS.
Uploaded 91 rows for H-107870 Hydroweb/Theia.
Uploaded 31523 rows for H-107870 GEOGLOWS.
Uploaded 124 rows for H-0000000007850 Hydroweb/Theia.
Uploaded 31523 rows for H-0000000007850 GEOGLOWS.
Uploaded 124 rows for H-101620 Hydroweb/Theia.
Uploaded 31523 rows for H-101620 GEOGLOWS.
Uploaded 104 rows for H-100136 Hydroweb/Theia.
Uploaded 31523 rows for H-100136 GEOGLOWS.
Uploaded 70 rows for H-107033 Hydroweb/Theia.
Uploaded 31523 rows for H-107033 GEOGLOWS.
Uploaded 89 rows for H-107872 Hydroweb/Theia.
Uploaded 31523 rows for H-107872 GEOGLOWS.
Uploaded 70 rows for H-109552 Hydroweb/Theia.
Uploaded 31523 rows for H-109552 GEOGLOWS.
Uploaded 70 rows for H-106712 Hydroweb/Theia.
Uploaded 31523 rows for H-106712 GEOGLOWS.
Uploaded 70 rows for H-101618 Hydroweb/Theia.
Uploaded 31523 rows for H-101618 GEOGLOWS.
Uploaded 69 rows for H-101183 Hydroweb/Theia.
Uploaded 31303 rows for H-101183 GEOGLOWS.
Uplo

,station_id,source,datastream_uuid,rows_uploaded,first_time,last_time
0,H-102549,Hydroweb/Theia,019dd0fe-33e4-7761-a045-e5aae79620ce,63,2020-09-02 00:00:00+00:00,2025-06-23 00:00:00+00:00
1,H-102549,GEOGLOWS,019dd0fe-3680-78ff-8206-331ee304f580,31523,1940-01-01 00:00:00+00:00,2026-04-21 00:00:00+00:00
2,H-107870,Hydroweb/Theia,019dd0fe-3d8f-757f-ad7c-bcb4ce6c241c,91,2018-12-31 00:00:00+00:00,2025-09-22 00:00:00+00:00
3,H-107870,GEOGLOWS,019dd0fe-402b-77ea-b82d-7f9ddb1b63a0,31523,1940-01-01 00:00:00+00:00,2026-04-21 00:00:00+00:00
4,H-0000000007850,Hydroweb/Theia,019dd0fe-4741-7c96-b237-aad670bda19b,124,2016-08-18 00:00:00+00:00,2025-09-21 00:00:00+00:00
...,...,...,...,...,...,...
71,H-108721,GEOGLOWS,019dd0ff-9e0b-7c38-83b8-bc3f1b86e3db,31523,1940-01-01 00:00:00+00:00,2026-04-21 00:00:00+00:00
72,H-0000000007959,Hydroweb/Theia,019dd0ff-a442-7102-a97f-1e128be39678,89,2019-01-04 00:00:00+00:00,2025-08-30 00:00:00+00:00
73,H-0000000007959,GEOGLOWS,019dd0ff-a7cd-7733-aba6-c9ad4673ac57,31523,1940-01-01 00:00:00+00:00,2026-04-21 00:00:00+00:00
74,H-108720,Hydroweb/Theia,019dd0ff-adf3-7a76-872d-cc62d881676e,88,2019-01-04 00:00:00+00:00,2025-08-30 00:00:00+00:00


## Cleanup: Delete Created Resources

In [ ]:
cleanup_order = [
    "task",
    "data_connection",
    "orchestration_system",
    "datastream",
    "thing",
    "result_qualifier",
    "processing_level",
    "sensor",
    "unit",
    "observed_property",
    "workspace",
]

preview = created_resources_dataframe()
if not preview.empty:
    display(preview)
else:
    print("No created resources are recorded.")

if not DELETE_CREATED_RESOURCES_AT_END:
    print("Cleanup skipped because DELETE_CREATED_RESOURCES_AT_END is False.")
elif hs_api is None:
    print("Cleanup skipped because the HydroServer client is unavailable.")
else:
    deleted_ids = set()
    for resource_type in cleanup_order:
        for row in reversed(created_resources):
            if row["resource_type"] != resource_type:
                continue
            resource = row["resource"]
            uid = resource_uid(resource)
            if resource is None or uid in deleted_ids:
                continue
            try:
                print(f"Deleting {resource_type}: {row['name_or_code']} | uuid={uid}")
                resource.delete()
                deleted_ids.add(uid)
            except Exception as exc:
                print(f"Could not delete {resource_type} {uid}: {exc}")
    print("Cleanup finished.")